In [3]:
# =====================================================================
# SYSTEM INITIALIZATION & DEPENDENCY SETUP
# =====================================================================
!pip install mlflow pydantic xgboost -q

import mlflow
import pydantic
import sklearn
import pandas as pd
import numpy as np

# Enterprise Solution: Route metadata tracking through a structured local SQLite DB
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("SaaS_Retention_Optimization_Engine")

print("--- System Environment Verified ---")
print(f"MLflow Tracking: Active (SQLite Database Engine)")
print(f"Pydantic Version: {pydantic.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026/06/14 11:13:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/14 11:13:34 INFO mlflow.store.db.utils: Updating database tables
2026/06/14 11:13:36 INFO mlflow.tracking.fluent: Experiment with name 'SaaS_Retention_Optimization_Engine' does not exist. Creating a new experiment.


--- System Environment Verified ---
MLflow Tracking: Active (SQLite Database Engine)
Pydantic Version: 2.12.3


# Enterprise Customer Churn & Retention Optimization Engine

### 1. Business Context & Strategic Impact
Acquiring a new customer costs subscription-based platforms significantly more than retaining an existing one. If an enterprise can predict which user profiles are showing early patterns of leaving (churning), retention teams can intervene with targeted incentives.

### 2. Engineering Objectives
This notebook creates an end-to-end predictive retention model using a massive consumer portfolio dataset. To ensure this code integrates perfectly into our upcoming multi-container Docker architecture, we implement:
* **Early Identifier Isolation:** Explicitly stripping out database tracking artifacts (Customer IDs and Surnames) before they can cause model over-fitting.
* **Strict Runtime Type Contracts:** A comprehensive Pydantic gateway to validate incoming web requests on the fly.
* **Audited Model Lineage:** Full MLflow logging tracking model evaluation metrics, training hyper-parameters, and pipeline configurations into a SQL database.

In [2]:
# =====================================================================
# DATA ACQUISITION & DATABASE TRAIN/TEST SPLITTING (UPDATED WORKING MIRROR)
# =====================================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import files

# Enterprise Solution: Replaced the broken 404 URL with a reliable public dataset mirror
data_url = "https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling.csv"
raw_retention_data = pd.read_csv(data_url)
raw_retention_data.to_csv('Churn_Modelling.csv', index=False)
files.download('Churn_Modelling.csv')


print("--- Raw Customer Data Structural Audit ---")
print(f"Total Customer Profiles Analysed: {raw_retention_data.shape[0]}")
print(f"Total Database Behavioral Features: {raw_retention_data.shape[1]}")

# DATA HYGIENE: Drop metadata artifacts that provide zero statistical signal
# This prevents our pipeline from accidentally relying on random database row sequences
cleaned_data = raw_retention_data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# Separate features from our target retention label ('Exited' -> 1 = Churned, 0 = Retained)
X = cleaned_data.drop(columns=['Exited'])
y = cleaned_data['Exited']

# Split partitions using stratified sampling to preserve customer churn ratios uniformly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining Partition Shape: {X_train.shape}")
print(f"Testing Partition Shape: {X_test.shape}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

--- Raw Customer Data Structural Audit ---
Total Customer Profiles Analysed: 10000
Total Database Behavioral Features: 14

Training Partition Shape: (8000, 10)
Testing Partition Shape: (2000, 10)


## 3. Formulating the Edge API Gateway (Pydantic Data Contract)
Before constructing the data transformations, we build our strict data validation schema. This model dictates the exact interface parameters the Flask user interface will accept, forcing strict constraints on attributes like consumer credit scores, geographical markets, and active status flags.

In [4]:
# =====================================================================
# PYDANTIC RUNTIME RETENTION SCHEMA GATEWAY
# =====================================================================
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class CustomerBehaviorSchema(BaseModel):
    # Enforce structural type limits matching real consumer scenarios
    CreditScore: int = Field(..., ge=300, le=850, description="Standard financial credit boundary.")
    Geography: Literal['France', 'Spain', 'Germany']
    Gender: Literal['Female', 'Male']
    Age: int = Field(..., ge=18, le=110)
    Tenure: int = Field(..., ge=0, le=10)
    Balance: float = Field(..., ge=0.0)
    NumOfProducts: int = Field(..., ge=1, le=4)
    HasCrCard: Literal[0, 1]
    IsActiveMember: Literal[0, 1]
    EstimatedSalary: float = Field(..., ge=0.0)

    @field_validator('EstimatedSalary')
    @classmethod
    def validate_positive_earnings(cls, value: float) -> float:
        if value < 0:
            raise ValueError("Estimated compensation values cannot be negative.")
        return value

print("Pydantic Customer Behavior Validation Schema safely compiled!")

Pydantic Customer Behavior Validation Schema safely compiled!


## 4. Multi-Branch Isolation Transformers
Consumer behavior models often combine sparse categorical markets with highly continuous variables like salaries and balances. We build an immutable `ColumnTransformer` to handle scaling and text encoding independently. This system completely insulates our model against missing categories or shape errors when handling individual user form entries.

In [5]:
# =====================================================================
# MULTI-CHANNEL PREPROCESSING MATRICES
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Isolate feature lanes based on data type rules
numerical_behavior_metrics = [
    'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary'
]
categorical_behavior_metrics = [
    'Geography', 'Gender', 'HasCrCard', 'IsActiveMember'
]

# Standardize mathematical attributes
numerical_pipeline = Pipeline(steps=[
    ('median_imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Process text descriptions safely into categorical arrays
categorical_pipeline = Pipeline(steps=[
    ('string_imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

# Fuse parallel transformation pathways into a single preprocessing gate
production_preprocessing_gate = ColumnTransformer(transformers=[
    ('num_channel', numerical_pipeline, numerical_behavior_metrics),
    ('cat_channel', categorical_pipeline, categorical_behavior_metrics)
])

print("Preprocessing pipeline securely locked down.")

Preprocessing pipeline securely locked down.


## 5. Instrumented Model Optimization & Metric Auditing (MLflow)
We pair our transformation mechanics with an optimized tree-boosting engine (**XGBoost**). The training execution runs inside an active **MLflow tracking manager**, mapping parameter adjustments and outputting precise precision-recall performance reports straight to our SQL experiment tracking log.

In [6]:
# =====================================================================
# INSTRUMENTED BOOSTING EXECUTION & AUDIT LINEAGE
# =====================================================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow.sklearn

# Calculate class distribution weights to balance the loss optimization natively (~4x imbalance ratio)
imbalance_ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

# Compile the final operational system blueprint
retention_optimization_pipeline = Pipeline(steps=[
    ('preprocessing_gate', production_preprocessing_gate),
    ('xgboost_engine', XGBClassifier(
        n_estimators=180,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=imbalance_ratio,
        random_state=42,
        n_jobs=-1
    ))
])

# Activate automatic pipeline telemetry tracking via MLflow
mlflow.sklearn.autolog(log_models=True)

print("Initializing training run. Recording telemetry inside MLflow tracker...")
with mlflow.start_run(run_name="Customer_Churn_XGBoost_Production") as active_run:

    # Train the comprehensive pipeline model
    retention_optimization_pipeline.fit(X_train, y_train)

    # Generate verification sets for testing
    predictions = retention_optimization_pipeline.predict(X_test)
    probabilities = retention_optimization_pipeline.predict_proba(X_test)[:, 1]

    # Calculate performance metrics
    test_roc_auc = roc_auc_score(y_test, probabilities)

    # Log specific custom retention insights directly to the dashboard
    mlflow.log_metric("production_test_roc_auc", test_roc_auc)

    print("\n=====================================================================")
    print("--- MLFLOW TRANSACTIONAL AUDIT COMPLETE ---")
    print(f"Logged SQL Run ID: {active_run.info.run_id}")
    print(f"Production ROC-AUC Score: {test_roc_auc:.4f}")
    print("=====================================================================\n")
    print(classification_report(y_test, predictions))

Initializing training run. Recording telemetry inside MLflow tracker...


2026/06/14 11:14:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/14 11:14:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils


--- MLFLOW TRANSACTIONAL AUDIT COMPLETE ---
Logged SQL Run ID: dac6f41598344ed9b01a14209970529f
Production ROC-AUC Score: 0.8668

              precision    recall  f1-score   support

           0       0.92      0.82      0.87      1593
           1       0.51      0.74      0.60       407

    accuracy                           0.80      2000
   macro avg       0.72      0.78      0.73      2000
weighted avg       0.84      0.80      0.81      2000



In [7]:
# =====================================================================
# HARD DRIVE SERIALIZATION EXPORT PROTOCOL
# =====================================================================
from google.colab import files
import joblib

# Dump the fully operational end-to-end pipeline object
joblib.dump(retention_optimization_pipeline, 'customer_churn_pipeline.pkl')

# Trigger direct local file download to your machine
files.download('customer_churn_pipeline.pkl')
print("Successfully generated and downloaded 'customer_churn_pipeline.pkl'!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully generated and downloaded 'customer_churn_pipeline.pkl'!
